In [1]:
!nvidia-smi

Fri Oct  9 21:32:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(1024, 1024, device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    print("GPU computation passed:", y.shape)
else:
    print("GPU unavailable — do not proceed.")

PyTorch: 2.11.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: Tesla T4
GPU computation passed: torch.Size([1024, 1024])


In [4]:
from google.colab import files
uploaded = files.upload()

Saving phase7_source.tar.gz to phase7_source.tar.gz


In [5]:
!sha256sum /content/phase7_source.tar.gz

f20581a4cb9b74ee8967485c259999f797fac0bbd44309ca1552aaaa2418c9f5  /content/phase7_source.tar.gz


In [6]:
!nvidia-smi
!nvcc --version
!g++ --version
!cmake --version

Fri Oct  9 21:40:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             28W /   70W |     137MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

    x = torch.randn(1024, 1024, device="cuda")
    y = x @ x
    torch.cuda.synchronize()

    print("GPU computation passed:", y.shape)
else:
    print("GPU unavailable — stop here.")

PyTorch: 2.11.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
GPU computation passed: torch.Size([1024, 1024])


In [8]:
import hashlib
import tarfile
from pathlib import Path

archive = Path("/content/phase7_source.tar.gz")
expected = "f20581a4cb9b74ee8967485c259999f797fac0bbd44309ca1552aaaa2418c9f5"

assert archive.is_file(), f"Archive not found: {archive}"

actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print("SHA-256:", actual)
assert actual == expected, "HASH MISMATCH — do not extract."

with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()
    print("Archive members:", len(members))
    print("First 15 paths:")
    for member in members[:15]:
        print(member.name)

print("Archive hash verified.")

SHA-256: f20581a4cb9b74ee8967485c259999f797fac0bbd44309ca1552aaaa2418c9f5
Archive members: 127
First 15 paths:
modern-algorithm-benchmark/AGENTS.md
modern-algorithm-benchmark/CMakeLists.txt
modern-algorithm-benchmark/README.md
modern-algorithm-benchmark/cmake/benchmark_metadata.cmake
modern-algorithm-benchmark/cmake/cuda.cmake
modern-algorithm-benchmark/docs/ALGORITHMS.md
modern-algorithm-benchmark/docs/AUDIT.md
modern-algorithm-benchmark/docs/BENCHMARKING.md
modern-algorithm-benchmark/docs/CUDA_EXECUTION_GUIDE_v1.md
modern-algorithm-benchmark/docs/CUDA_PREFLIGHT_PROTOCOL_v1.md
modern-algorithm-benchmark/docs/EXPERIMENT_PROTOCOL.md
modern-algorithm-benchmark/docs/METRICS.md
modern-algorithm-benchmark/docs/OPENMP_PROTOCOL_v1.md
modern-algorithm-benchmark/docs/ORCHESTRATION_PROTOCOL_v1.md
modern-algorithm-benchmark/docs/ROADMAP.md
Archive hash verified.


In [9]:
import tarfile
from pathlib import Path, PurePosixPath

archive = Path("/content/phase7_source.tar.gz")
target = Path("/content/modern-algorithm-benchmark")

assert archive.is_file(), "Archive not found."
assert not target.exists(), (
    f"{target} already exists. Inspect it before extracting again."
)

with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()

    for member in members:
        path = PurePosixPath(member.name)

        assert not path.is_absolute(), f"Absolute path rejected: {member.name}"
        assert ".." not in path.parts, f"Path traversal rejected: {member.name}"
        assert path.parts[0] == "modern-algorithm-benchmark", (
            f"Unexpected root path: {member.name}"
        )
        assert not (member.issym() or member.islnk()), (
            f"Link rejected: {member.name}"
        )

    assert len(members) == 127, f"Unexpected member count: {len(members)}"

    # Python's data filter adds another layer of extraction protection.
    tar.extractall("/content", members=members, filter="data")

print("Extraction completed.")
print("Project directory:", target)
print("CUDA guide exists:", (target / "docs/CUDA_EXECUTION_GUIDE_v1.md").is_file())
print("Source manifest exists:", (target / "SOURCE_SHA256.json").is_file())

Extraction completed.
Project directory: /content/modern-algorithm-benchmark
CUDA guide exists: True
Source manifest exists: True


In [10]:
from pathlib import Path

guide = Path(
    "/content/modern-algorithm-benchmark/docs/CUDA_EXECUTION_GUIDE_v1.md"
)

assert guide.is_file(), f"CUDA guide not found: {guide}"
print(guide.read_text())

# CUDA preflight and Colab/cloud guide v1

This guide prepares future GPU work. **No CUDA sorter or GPU timing exists yet.**
The local validation machine has an NVIDIA PCI device, but no discoverable nvcc,
nvidia-smi or working CUDA driver evidence. CUDA target compilation, kernel
execution and the Colab workflow have **not been validated on a GPU**. Mocked
readiness results are tests, never hardware evidence. The existing CPU studies
remain frozen. Follow [the readiness contract](CUDA_PREFLIGHT_PROTOCOL_v1.md).

## 1. Allocate and verify an environment

In Google Colab select Runtime → Change runtime type → an available GPU hardware
accelerator, then connect. A selection is not evidence a GPU was allocated;
[Colab does not guarantee particular resources](https://research.google.com/colaboratory/faq.html).
Equivalent cloud environments need an NVIDIA GPU instance and provider-supported
CUDA image. Do not upgrade or replace hosted drivers from this guide.

Run discovery in a notebook `%

In [11]:
import hashlib
import json
from pathlib import Path

root = Path("/content/modern-algorithm-benchmark")
manifest_path = root / "SOURCE_SHA256.json"

assert root.is_dir(), f"Project directory not found: {root}"
assert manifest_path.is_file(), "Source manifest missing"

manifest = json.loads(manifest_path.read_text())
failures = []

for relative_path, expected_hash in manifest.items():
    path = root / relative_path

    if not path.is_file():
        failures.append(f"Missing: {relative_path}")
        continue

    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        failures.append(f"Hash mismatch: {relative_path}")

if failures:
    print("\n".join(failures))
    raise RuntimeError("Source integrity verification failed")

print(f"Source integrity verified: {len(manifest)} files")

Source integrity verified: 126 files


In [14]:

%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

echo "=== Python and pip ==="
python3 --version
python3 -m pip --version

echo "=== Required packages ==="
python3 -c "import numpy, matplotlib; print('NumPy:', numpy.__version__); print('Matplotlib:', matplotlib.__version__)"

echo "=== CUDA preflight script ==="
test -f scripts/cuda_preflight.py
echo "Preflight script exists"

echo "=== Create output directories ==="
mkdir -p results/raw results/processed results/figures

echo "Environment checks passed."


=== Python and pip ===
Python 3.13.15
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
=== Required packages ===
NumPy: 2.1.3
Matplotlib: 3.10.0
=== CUDA preflight script ===
Preflight script exists
=== Create output directories ===
Environment checks passed.


In [15]:

import datetime
import pathlib
import subprocess
import sys

root = pathlib.Path("/content/modern-algorithm-benchmark")
stamp = datetime.datetime.now(
    datetime.timezone.utc
).strftime("%Y%m%dT%H%M%S%fZ")

run = root / "results/raw" / f"cloud_cuda_preflight_{stamp}"

result = subprocess.run(
    [
        sys.executable,
        str(root / "scripts/cuda_preflight.py"),
        "--output",
        str(run),
        "--timeout-seconds",
        "180",
    ],
    cwd=root,
    text=True,
)

print("Preflight exit code:", result.returncode)
print("Evidence directory:", run)

for name in ("SUMMARY.txt", "report.json"):
    path = run / name
    print(f"\n--- {name} ---")
    print(path.read_text() if path.exists() else "FILE NOT FOUND")


Preflight exit code: 0
Evidence directory: /content/modern-algorithm-benchmark/results/raw/cloud_cuda_preflight_20261009T214717153394Z

--- SUMMARY.txt ---
CUDA readiness: operational
Device0 kernel verified: True
NVIDIA nvcc answers: True
Project CUDA target built: True
CPU project: unaffected by unavailable CUDA; use MAB_ENABLE_CUDA=OFF.
Visible to this process/container; absence of detection does not prove host GPU absence.
nvidia_smi: completed; exit=0
lspci: unavailable; exit=None
nvcc: completed; exit=0
cmake: completed; exit=0
cmake_cuda_property: completed; exit=0
configure: completed; exit=0
build: completed; exit=0
runtime: completed; exit=0
NVIDIA GPU: Tesla T4; driver=580.82.07
No installation/system changes; no sorting/performance experiment.
Only device0 kernel execution is checked; other enumerated devices untested.
Toolkit/Colab GPU path unvalidated unless operational evidence is present.


--- report.json ---
{
  "captured_at_utc": "2026-10-09T21:47:24.948424+00:00",
 

In [16]:

%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

cmake -S . -B build/cloud_cuda \
  -DCMAKE_BUILD_TYPE=Release \
  -DMAB_ENABLE_CUDA=ON \
  -DMAB_ENABLE_OPENMP=OFF \
  -DMAB_ENABLE_SANITIZERS=OFF \
  -DCMAKE_CUDA_ARCHITECTURES=75

cmake --build build/cloud_cuda --target cuda_runtime_probe --parallel 2

build/cloud_cuda/cuda_runtime_probe

ctest --test-dir build/cloud_cuda \
  --output-on-failure -V -R '^cuda_runtime_smoke$'


-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /usr/local/bin/python (found version "3.13.15") found components: Interpreter
-- Looking for a CUDA compiler
-- Looking for a CUDA compiler - /usr/local/cuda/bin/nvcc
-- The CUDA compiler identification is NVIDIA 13.0.88 with host compiler GNU 13.3.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include;/usr/local/cuda/targets/x86_64-linux/include/cccl (found suitable version "13.0.88", minimum required is "11.0")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMA

In [17]:

%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

cmake -S . -B build/cloud_cpu \
  -DCMAKE_BUILD_TYPE=Release \
  -DMAB_ENABLE_CUDA=OFF \
  -DMAB_ENABLE_OPENMP=OFF

cmake --build build/cloud_cpu --parallel 2

ctest --test-dir build/cloud_cpu --output-on-failure -V


-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /usr/local/bin/python (found version "3.13.15") found components: Interpreter
-- Configuring done (0.6s)
-- Generating done (0.1s)
-- Build files have been written to: /content/modern-algorithm-benchmark/build/cloud_cpu
[  1%] Building CXX object CMakeFiles/mab_core.dir/src/algorithms/baselines.cpp.o
[  2%] Building CXX object CMakeFiles/mab_core.dir/src/datasets/generator.cpp.o
[  4%] Building CXX object CMakeFiles/mab_core.dir/src/algorithms/quicksort.cpp.o
[  5%] Building CXX object CMakeFiles/mab_core.dir/src/algorithms/sorting.cpp.o
[  6%] Building CXX object CMakeFiles/mab_core.dir/src/algorithms/pdqsort.cpp.o
[  8%] Building CXX object CMakeFiles/mab_core.dir/src/algorithms/timsort.cpp.o
[  9%] Building CXX

In [18]:

%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

STAMP="$(date -u +%Y%m%dT%H%M%SZ)"
OUT="results/raw/cloud_cpu_environment_${STAMP}.json"

python3 scripts/capture_environment.py \
  --build-dir build/cloud_cpu \
  --output "$OUT"

echo "Environment record: $OUT"
test -s "$OUT"
cat "$OUT"


results/raw/cloud_cpu_environment_20261009T215305Z.json
Environment record: results/raw/cloud_cpu_environment_20261009T215305Z.json
{
  "build_directory": "/content/modern-algorithm-benchmark/build/cloud_cpu",
  "captured_at_utc": "2026-10-09T21:53:05.647610+00:00",
  "cmake": {
    "command": [
      "cmake",
      "--version"
    ],
    "returncode": 0,
    "stderr": "",
    "stdout": "cmake version 3.31.10\n\nCMake suite maintained and supported by Kitware (kitware.com/cmake).\n"
  },
  "cmake_cache": "# This is the CMakeCache file.\n# For build in directory: /content/modern-algorithm-benchmark/build/cloud_cpu\n# It was generated by CMake: /usr/local/lib/python3.13/dist-packages/cmake/data/bin/cmake\n# You can edit this file to change values found and used by cmake.\n# If you do not want to change any of the values, simply exit the editor.\n# If you do want to change a value, simply edit, save, and exit the editor.\n# The syntax for the file is as follows:\n# KEY:TYPE=VALUE\n# KEY i

In [20]:
%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

STAMP="$(date -u +%Y%m%dT%H%M%SZ)"
OUT="results/raw/cloud_cpu_environment_${STAMP}.json"

python3 scripts/capture_environment.py \
  --build-dir build/cloud_cpu \
  --output "$OUT"

echo "Environment record: $OUT"
test -s "$OUT"
cat "$OUT"

results/raw/cloud_cpu_environment_20261009T215408Z.json
Environment record: results/raw/cloud_cpu_environment_20261009T215408Z.json
{
  "build_directory": "/content/modern-algorithm-benchmark/build/cloud_cpu",
  "captured_at_utc": "2026-10-09T21:54:08.438612+00:00",
  "cmake": {
    "command": [
      "cmake",
      "--version"
    ],
    "returncode": 0,
    "stderr": "",
    "stdout": "cmake version 3.31.10\n\nCMake suite maintained and supported by Kitware (kitware.com/cmake).\n"
  },
  "cmake_cache": "# This is the CMakeCache file.\n# For build in directory: /content/modern-algorithm-benchmark/build/cloud_cpu\n# It was generated by CMake: /usr/local/lib/python3.13/dist-packages/cmake/data/bin/cmake\n# You can edit this file to change values found and used by cmake.\n# If you do not want to change any of the values, simply exit the editor.\n# If you do want to change a value, simply edit, save, and exit the editor.\n# The syntax for the file is as follows:\n# KEY:TYPE=VALUE\n# KEY i

In [21]:
%%bash
set -euo pipefail
cd /content/modern-algorithm-benchmark

mkdir -p results/evidence

tar -czf results/evidence/cloud_validation_evidence.tar.gz \
  results/raw/cloud_cpu_environment_20261009T215408Z.json \
  results/raw/cloud_cuda_preflight_20261009T214717153394Z \
  build/cloud_cpu/Testing/Temporary/LastTest.log \
  build/cloud_cuda/Testing/Temporary/LastTest.log

sha256sum results/evidence/cloud_validation_evidence.tar.gz
ls -lh results/evidence/cloud_validation_evidence.tar.gz

48b3e78360e7c0c1721c21a3879bc975b902f5b3603fa25a0c92230c695c0810  results/evidence/cloud_validation_evidence.tar.gz
-rw-r--r-- 1 root root 14K Oct  9 21:54 results/evidence/cloud_validation_evidence.tar.gz


In [26]:
from pathlib import Path
from google.colab import files

# Check the expected location first
candidates = [
    Path("/content/modern-algorithm-benchmark/results/evidence/cloud_validation_evidence.tar.gz"),
    Path("/content/results/evidence/cloud_validation_evidence.tar.gz"),
    Path("results/evidence/cloud_validation_evidence.tar.gz"),
]

archive = next((p for p in candidates if p.is_file()), None)

# Search /content if it wasn't found at the expected paths
if archive is None:
    matches = list(Path("/content").rglob("cloud_validation_evidence.tar.gz"))
    if matches:
        archive = matches[0]

if archive is None:
    print("Archive not found. Current directory:", Path.cwd())
    print("Check whether the runtime restarted or the archive was created in another session.")
else:
    print("Found:", archive.resolve())
    print("Size:", archive.stat().st_size, "bytes")
    files.download(str(archive.resolve()))

Found: /content/modern-algorithm-benchmark/results/evidence/cloud_validation_evidence.tar.gz
Size: 13534 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>